In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 78.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=711956be51e6f03cd208a59632b61c7a8bdc1ab6d63c69137f7c9d87be03c389
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [3]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [7]:
# Step 1: Helper function to generate quantum random bits

def quantum_random_bit():

    # Create a quantum circuit with 1 qubit and 1 classical bit
    qc = QuantumCircuit(1, 1)

    # Put qubit into superposition
    qc.h(0)

    # Measure the qubit
    qc.measure(0, 0)

    # Use simulator
    simulator = BasicSimulator()

    # Run circuit
    result = simulator.run(qc, shots=1).result()

    # Get measurement counts
    counts = result.get_counts()

    # Return measured bit
    bit = list(counts.keys())[0]

    return int(bit)

In [8]:
# Test the quantum random generator

for i in range(10):
    print(quantum_random_bit())

1
1
1
0
1
1
1
1
0
0


In [9]:
# Step 2: Generate sender bits and sender bases

number_of_bits = 10

sender_bits = []
sender_bases = []

for i in range(number_of_bits):

    # Random secret bit
    bit = quantum_random_bit()

    # Random basis
    # 0 = Z basis (+)
    # 1 = X basis (x)
    basis = quantum_random_bit()

    sender_bits.append(bit)
    sender_bases.append(basis)

print("Sender bits: ", sender_bits)
print("Sender bases:", sender_bases)

Sender bits:  [0, 1, 1, 0, 0, 1, 1, 0, 0, 1]
Sender bases: [0, 1, 1, 1, 1, 0, 1, 1, 0, 1]


In [10]:
# Step 3: Sender prepares qubits according to bits and bases

prepared_qubits = []

for i in range(number_of_bits):

    qc = QuantumCircuit(1, 1)

    # If bit is 1, flip |0> to |1>
    if sender_bits[i] == 1:
        qc.x(0)

    # If basis is X basis, apply Hadamard
    if sender_bases[i] == 1:
        qc.h(0)

    prepared_qubits.append(qc)

print("Qubits prepared successfully.")

Qubits prepared successfully.


In [11]:
# Step 4: Receiver generates random bases

receiver_bases = []

for i in range(number_of_bits):

    basis = quantum_random_bit()

    receiver_bases.append(basis)

print("Receiver bases:", receiver_bases)

Receiver bases: [1, 0, 1, 0, 0, 1, 0, 0, 0, 1]


In [12]:
# Step 5: Receiver measures the qubits

receiver_results = []

simulator = BasicSimulator()

for i in range(number_of_bits):

    qc = prepared_qubits[i].copy()

    # If receiver chooses X basis, apply Hadamard before measuring
    if receiver_bases[i] == 1:
        qc.h(0)

    qc.measure(0, 0)

    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()
    measured_bit = int(list(counts.keys())[0])

    receiver_results.append(measured_bit)

print("Receiver results:", receiver_results)

Receiver results: [1, 1, 1, 1, 1, 0, 0, 1, 0, 1]


In [13]:
# Step 6: Compare bases and generate shared key

shared_key_sender = []
shared_key_receiver = []

for i in range(number_of_bits):

    # Keep bits only if bases match
    if sender_bases[i] == receiver_bases[i]:

        shared_key_sender.append(sender_bits[i])
        shared_key_receiver.append(receiver_results[i])

print("Sender key:  ", shared_key_sender)
print("Receiver key:", shared_key_receiver)

Sender key:   [1, 0, 1]
Receiver key: [1, 0, 1]


In [14]:
# Step 7: Verify whether the keys match

if shared_key_sender == shared_key_receiver:
    print("BB84 protocol successful.")
    print("Shared secret key established.")
else:
    print("Keys do not match.")

BB84 protocol successful.
Shared secret key established.
